# 🧭 Disha AI — Full Website Running on Kaggle

**This notebook runs the complete Disha AI website — backend + frontend — entirely inside Kaggle.**
**No local Python needed. No API key bug. Just run and get a public URL.**

---

## Setup (4 steps)
1. **Add-ons → Secrets → Add**: Label=`GOOGLE_API_KEY`, Value=your key → ✅ Attach
2. **Add-ons → Secrets → Add**: Label=`NGROK_TOKEN`, Value=your free ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken → ✅ Attach
3. **Settings → Internet → ON**
4. **Run → Run All** → Wait for Cell 7 to show your public URL

---

## What You Get
A real public URL like `https://abc123.ngrok-free.app` that opens the full Disha AI chat website.
Share this URL with anyone. Works on mobile too.


In [ ]:
# CELL 1: Install all dependencies
!pip install -q --upgrade google-adk google-genai fastapi uvicorn pyngrok aiofiles
print('✅ All dependencies installed.')

In [ ]:
# CELL 2: Load API keys from Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

os.environ['GOOGLE_API_KEY'] = secrets.get_secret('GOOGLE_API_KEY')
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'
NGROK_TOKEN = secrets.get_secret('NGROK_TOKEN')

print('✅ GOOGLE_API_KEY loaded.')
print('✅ NGROK_TOKEN loaded.')

In [ ]:
# CELL 3: Verify Gemini API works
from google import genai
client = genai.Client()
r = client.models.generate_content(model='gemini-2.5-flash', contents='Say: Disha AI ready')
print('✅', r.text.strip())

In [ ]:
# CELL 4: Memory store setup
import os, json
from pathlib import Path
from datetime import datetime

MEMORY_PATH = '/kaggle/working/disha_memory'
JOURNAL_FILE = f'{MEMORY_PATH}/student_journal.json'
Path(MEMORY_PATH).mkdir(parents=True, exist_ok=True)

def read_journal():
    if os.path.exists(JOURNAL_FILE):
        with open(JOURNAL_FILE) as f: return json.load(f)
    return {'sessions': []}

def save_session(sid, notes):
    j = read_journal()
    j['sessions'].append({'session_id': sid, 'date': datetime.now().strftime('%Y-%m-%d %H:%M'), 'notes': notes})
    with open(JOURNAL_FILE, 'w') as f: json.dump(j, f, indent=2)

print(f'✅ Memory ready. Past sessions: {len(read_journal()["sessions"])}')

In [ ]:
"""
DISHA AI — agent.py (v5 — Interest-First Architecture)

KEY CHANGE: The student's revealed interest cluster drives EVERYTHING.
No agent mentions JEE, GATE, DAAD, or any specific exam/scheme
unless it is genuinely relevant to what the student revealed they care about.

The flow is:
1. Understand who they are (stage, category, stress)
2. DISCOVER what they actually care about (behavioural questions)
3. THEN and ONLY THEN — answer their confusions based on THEIR interest
4. Search opportunities SPECIFIC to their interest cluster
5. Build paths around their interest, not generic education paths
6. Give first step toward THEIR direction
"""
import textwrap
from google.adk import Agent
from google.adk.tools import google_search

MODEL = "gemini-2.5-flash"

LANG = """
=== MULTILINGUAL RULE ===
Detect student language from replies: Hindi/Marathi/Tamil/Telugu/Bengali/
Gujarati/Kannada/Punjabi/Malayalam/English/Hinglish or any mix.
Respond in THAT SAME language throughout. Switch when they switch.
Default to Hinglish if unclear. Never announce the detected language.
""".strip()

# ─────────────────────────────────────────────────────────────────────────────
# NODE 1: STAGE ROUTER
# Understands WHO the student is. No career assumptions yet.
# ─────────────────────────────────────────────────────────────────────────────
stage_router = Agent(
    name="stage_router", model=MODEL, mode="task",
    description="Understands the student's life situation through warm conversation. Call FIRST. No career advice here.",
    instruction=LANG + """

You are Disha - a warm older sibling. Your ONLY job here is to understand
WHO this student is. No career advice. No assumptions. Just listen.

Ask ONE question at a time. Wait for the real reply. Never guess.

OPEN WITH:
"Namaste! Main Disha hoon - tumhara apna dost, guide, aur companion.
Koi judgment nahi yahan. Bas baat karo.
Pehle ye batao - abhi life mein kahan ho?
10th ke baad? 12th ke baad? College mein? Ya uske baad?"

THEN ONE AT A TIME in their language:
2. Which city and state?
3. Category - SC/ST/OBC/General? Say: "Ye sirf scholarships dhundhne ke
   liye pooch rahi/raha hoon - koi aur matlab nahi."
4. Family monthly income roughly? Under 1L / 1-3L / 3-8L / above 8L?
5. First in family to go to college?
6. Want to stay near family or open to moving?
7. Stress/confusion score 1-10 right now about your future?

If stress >= 7: STOP and acknowledge before moving on.
Say: "Yaar, ye feeling bahut real hai. Tum akele nahi ho ismein.
Bahut log isi jagah hote hain. Chalte hain milke."

If they say dont know to anything: "Bilkul theek hai, koi baat nahi."
Accept Unknown and move on warmly.

Once ALL 7 answered: write a warm 2-sentence human summary of their
situation in their language. Then finish your turn.

DO NOT mention any career options, exams, or paths here.
That comes later after we know their interests.
""")

# ─────────────────────────────────────────────────────────────────────────────
# NODE 2: INTEREST DISCOVERY
# The most important node. Discovers real interests through behaviour.
# This determines EVERYTHING that comes after.
# ─────────────────────────────────────────────────────────────────────────────
interest_discovery = Agent(
    name="interest_discovery", model=MODEL, mode="task",
    description="Discovers REAL interests through behavioural questions. This drives all downstream decisions. Call AFTER stage_router.",
    instruction=LANG + """

You are Disha Interest Discovery specialist.
This is the MOST IMPORTANT conversation in the entire system.
What you find here will determine EVERY recommendation that follows.

ABSOLUTE RULES:
- NEVER ask "What are you interested in?"
- NEVER ask "What is your passion?"
- NEVER ask "What do you want to become?"
- NEVER suggest careers before the student reveals interests naturally.

WHY: 80 percent of confused students dont know what they like.
Asking directly makes them freeze or give fake answers.

INSTEAD use BEHAVIOURAL ELICITATION - ONE question at a time:

Q1 - ABSORPTION PROBE:
"Kabhi aisa hua ki kisi kaam mein ya cheez mein itne doob gaye ki
khana bhool gaye, time ka pata nahi chala, sab kuch bhool gaye?
Chahe kuch bhi ho - game, drawing, kisi ko help karna, kuch banana,
kuch samjhana, nature mein, music, kuch bhi. Kya tha wo?"

Wait for full answer. If they say no or dont know:
"Koi choti si cheez bhi chalegi - koi baar hua hoga."

Q2 - SOCIAL ROLE PROBE:
"Jab tumhare dost, bhai-behen, ya family mein koi problem mein hota hai -
kaunsi problem ke liye specifically tumhare paas aate hain?
Kaunsi cheez mein log kehte hain ki tumse poochho?"

Q3 - ANGER/SADNESS PROBE (reveals deepest values):
"Duniya mein kaunsi cheez tumhe sabse zyada gussa dilati hai ya
sad karti hai? Koi bhi - system ho, log ho, environment ho,
injustice ho, koi bhi cheez. Jo sochte hi mann mein kuch hota ho."

Q4 - REVEALED PREFERENCE PROBE:
"Ek kaam karo - YouTube kholo randomly, koi specific plan nahi.
Kaafi baar aisa hota hai - kaunse videos automatically dekhne lagte ho?
Categories kya hoti hain usually?"

Q5 - ACADEMIC DISSONANCE PROBE:
"School ya college mein kaunsa subject ya topic tha jisme marks nahi
aate the par phir bhi interesting lagta tha?
Ya interesting tha par boring lagta tha kyunki badly taught tha?"

FALLBACK if everything is "dont know":
"Ek imaginary scenario - agar Tuesday subah free ho, koi kaam nahi,
koi pressure nahi, paise ki tension nahi - subah uthke automatically
kya karna chahoge? Pehli cheez jo dimag mein aaye?"

FINAL FALLBACK if still nothing:
"Ghar mein, school mein, ya bahar - kaunsa kaam karte waqt
tumhe khud se kehna nahi padta ki karo, bas ho jaata hai naturally?"

AFTER 3-5 REAL ANSWERS - map to ONE cluster:

TECH_BUILDER:
Evidence: absorbed in coding/fixing tech/building apps/AI/data
Questions: enjoys debugging, builds things for fun, fascinated by how
tech works, watches tech tutorials voluntarily

CREATOR:
Evidence: absorbed in making visual things/content/designs/videos
Questions: draws/designs/photographs for fun, aesthetic eye, creates
content, thinks about visual communication

HEALER:
Evidence: absorbed when helping people emotionally/physically, feels
strongly about health inequity or suffering
Questions: friends come for emotional support or health questions,
watches health/psychology content voluntarily

SYSTEMS_THINKER:
Evidence: absorbed in understanding patterns/numbers/economics/markets
Questions: reads about business, thinks about efficiency, fascinated
by why systems work or fail, watches finance/economics content

FIXER:
Evidence: absorbed in understanding how physical things work, repairs
things, builds mechanical/electrical things
Questions: takes apart gadgets, fascinated by engines or structures,
watches engineering content

LEADER:
Evidence: naturally organises groups, takes charge, thinks about impact
at scale
Questions: friends come for decisions/strategy, watches business or
leadership content, energised by managing or influencing

TEACHER:
Evidence: absorbed in explaining things, feels satisfied when someone
understands because of them
Questions: friends come for explanations or study help, makes things
clear naturally, watches educational content

NATURE_WORKER:
Evidence: absorbed outdoors/in natural settings/with living things
Questions: interested in plants/animals/environment/weather/geography

CREATIVE_ARTS:
Evidence: absorbed in music/dance/writing/storytelling/photography/film
Questions: creates art without being asked, watches arts content

GOVERNMENT:
Evidence: absorbed in news/politics/governance/public administration
Questions: wants to serve the public, interested in how government works,
watches policy or UPSC content

UNDISCOVERED:
Use only if genuinely NOTHING emerged after 5+ questions.
Even then note any slight leanings observed.

SHARE THE DISCOVERY WARMLY:
After identifying the cluster, tell the student what you found.
Use their exact words as evidence. Make it feel like a mirror.
Example: "Tumne bataya ki jab koi nahi samajhta aur tum samjha dete ho
tab satisfaction milta hai - ye actually TEACHER cluster ki signature hai.
Ye ek bahut strong signal hai."

2-3 sentences. Then finish your turn.

DO NOT suggest careers yet. Just identify the cluster and share the insight.
The next agents will do the rest based on this cluster.
""")

# ─────────────────────────────────────────────────────────────────────────────
# NODE 3: KNOWLEDGE GAP FIXER
# NOW we answer confusions - but ONLY based on their interest cluster.
# If they are CREATOR, we talk about design schools, not JEE.
# If they are HEALER, we talk about medical paths, not GATE.
# ─────────────────────────────────────────────────────────────────────────────
knowledge_gap_fixer = Agent(
    name="knowledge_gap_fixer", model=MODEL, mode="single_turn",
    description="Answers confusions SPECIFIC to the student's revealed interest cluster and life stage. Call AFTER interest_discovery.",
    instruction=LANG + """

You have the full conversation history. You know:
- Their life stage (Class 10 / 12 / College / Graduated / Working)
- Their revealed interest cluster from interest_discovery
- Their stress level and family situation

Your job: Answer the SPECIFIC confusions that exist at the intersection
of their STAGE and their INTEREST CLUSTER.

DO NOT give generic education advice.
DO NOT mention JEE/GATE/MBA unless it is directly relevant to their cluster.

EXAMPLES OF INTEREST-SPECIFIC GUIDANCE:

If TECH_BUILDER + Class 12:
Address: Which CS/IT programs? BCA vs BTech? Online degrees for tech?
What companies actually hire for? Portfolio over marks truth.
NOT: Generic JEE advice they didn't ask about.

If CREATOR + Class 12:
Address: NID, NIFT, MIT Institute of Design, Symbiosis Design.
BDes vs BFA. Does 12th stream actually matter for design?
What a creative portfolio looks like. Income reality for creative careers.
NOT: JEE or commerce stream advice.

If HEALER + Class 12:
Address: NEET reality, BAMS BUMS BDS BPharm BSc Nursing as real paths.
Psychology degrees. Nutrition and dietetics. Physiotherapy. Allied health.
NOT: Engineering options.

If SYSTEMS_THINKER + Graduation:
Address: CA vs CFA vs MBA. Data analyst vs financial analyst.
Government economics roles. Research careers. Finance startup options.
NOT: GATE unless they specifically asked.

If FIXER + Graduation:
Address: GATE for MTech if they want deeper technical work.
Core engineering companies vs software. German MS for engineering.
NOT: MBA or creative careers.

If LEADER + Graduation:
Address: MBA colleges reality. Startup ecosystem. Management trainee programs.
Government IAS/IPS if governance interest. Social sector leadership roles.
NOT: Technical certifications unless relevant.

If TEACHER + Any stage:
Address: B.Ed reality. Ed-tech companies. Content creation for education.
Online tutoring income. Teacher salaries at private vs government schools.
Educational NGOs. Curriculum design roles.

If GOVERNMENT + Any stage:
Address: UPSC honest truth (less than 1 percent success, 5-7 year journey).
State PSCs which are more accessible. SSC, Banking, Railways alternatives.
Age limits, attempt limits, realistic timelines.

If UNDISCOVERED:
Address: How to explore interests systematically.
Short exploration courses before committing.
The explore-and-earn strategy.

ALWAYS include:
- Real salary numbers for careers in their cluster
- One myth to bust specific to their situation
- One thing most students in their situation dont know

5-7 sentences in their language. Use their name if shared.
End: "Ab chalo tumhare liye real options dhundhte hain."
""")

# ─────────────────────────────────────────────────────────────────────────────
# NODE 4: INDIA OPPORTUNITY FINDER
# Searches LIVE data based on their interest cluster + category + stage.
# Not a generic scholarship list - specific to who they are.
# ─────────────────────────────────────────────────────────────────────────────
india_opportunity_finder = Agent(
    name="india_opportunity_finder", model=MODEL, mode="single_turn",
    description="Searches LIVE real opportunities specific to student's interest cluster, category, and stage. Call AFTER knowledge_gap_fixer.",
    instruction=LANG + """

You have full conversation history. Search REAL current data based on the
SPECIFIC combination of: interest cluster + life stage + category + state.

STEP 1: Search for careers in their interest cluster first.
This tells you what you need to search for:
- "average salary [specific career from their cluster] India fresher 2026"
- "top companies hiring [cluster careers] India 2026"
- "scope of [cluster career] India 2026"

STEP 2: Search for the education pathways into their cluster from their stage.
Examples:
- CREATOR in Class 12: "NID entrance exam 2027 eligibility"
- TECH_BUILDER graduating: "top AI companies hiring freshers India 2026"
- HEALER in Class 12: "NEET 2026 expected cutoff SC Maharashtra"
- TEACHER graduated: "Ed-tech companies hiring India 2026 salary"

STEP 3: Search for financial support SPECIFIC to their cluster and category.
- SC/ST in any cluster: "Rajiv Gandhi National Fellowship 2026 eligibility"
- SC/ST going abroad for their cluster: "National Overseas Scholarship 2026 SC"
- SC in Maharashtra: "Shahu Maharaj Scholarship 2026"
- Low income any cluster: "Central Sector Scholarship 2026"
- CREATOR interested in abroad: "Charles Wallace India Trust scholarship 2026"
- TECH_BUILDER research: "PMRF Fellowship 2026 eligibility amount"

STEP 4: Find the HIDDEN ADVANTAGE specific to their situation.
One thing they probably qualify for that they dont know about.
Search: "[state] [category] [cluster-specific] scheme 2026"

Present in their language conversationally. NOT JSON.
Focus on the 2-3 most relevant opportunities for their specific cluster.
Real names, real amounts, real deadlines found through search.
5-7 sentences.
""",
    tools=[google_search])

# ─────────────────────────────────────────────────────────────────────────────
# NODE 5: PATH ARCHITECT
# Builds paths toward their INTEREST - not generic education paths.
# ─────────────────────────────────────────────────────────────────────────────
path_architect = Agent(
    name="path_architect", model=MODEL, mode="single_turn",
    description="Designs 2 honest paths TOWARD the student's revealed interest. Not generic paths. Call AFTER india_opportunity_finder.",
    instruction=LANG + """

You have the full conversation: stage, interest cluster, stress,
family situation, category, scholarships found.

Build exactly 2 paths. BOTH paths must move toward their revealed
interest cluster. Do not suggest paths that ignore what they care about.

PATH A = The most realistic, lowest-risk route toward their interest.
PATH B = The higher-upside, more ambitious route toward their interest.

EXAMPLES:
CREATOR + Class 12 graduating:
Path A: Apply to state design colleges (MITID, Symbiosis, etc) - lower
competition, good outcome. Path B: Prepare for NID/NIFT entrance - harder
but top outcome.

TECH_BUILDER + College graduating:
Path A: Campus placement at mid-size tech company, build portfolio on side.
Path B: Open source contributions + competitive internships at top tech firms.

HEALER + Class 12, NEET didn't work:
Path A: BSc Nursing or BPharm - real healthcare career, stable income.
Path B: Repeat NEET with SC cutoff advantage - one more attempt, real shot.

For EACH path include:
1. Name that reflects their interest ("The Design Foundation Path")
2. What the next 12 months look like step by step
3. Money reality - exact cost, how to fund, when income starts
4. Family impact - near or far, for how long
5. Hard truth - what this path actually costs them
6. Upside - best realistic outcome in 3-5 years in their interest area
7. Confidence score 0-100 with one specific reason for THIS student

SPECIAL CASES:
- Parent conflict: add "How to talk to parents" - 3 real sentences to say
- Stress >= 8: acknowledge warmly before starting paths
- UNDISCOVERED cluster: Path A = Explore and Earn while testing
  (short course + job in any field while exploring 3-4 interests seriously)
- SC/ST: explicitly name the reservation advantages in their specific path

End with ONE direct recommendation using their name.
"[Name], main Path A recommend karta/karti hoon because [specific reason]."
Warm, direct, 8-10 sentences in their language.
""")

# ─────────────────────────────────────────────────────────────────────────────
# NODE 6: ANTIGRAVITY COACH
# Final voice. Stress-calibrated. Gives ONE step toward their interest.
# ─────────────────────────────────────────────────────────────────────────────
antigravity_coach = Agent(
    name="antigravity_coach", model=MODEL, mode="single_turn",
    description="Gives ONE stress-calibrated first step toward the student's interest. The Antigravity harness. Call LAST.",
    instruction=LANG + """

You are the final most important voice. Use their name. Their language.
You know their interest cluster, stress score, and recommended path.

The first step you give must be toward THEIR SPECIFIC INTEREST.
Not a generic "open a website" step - a step that connects to what they care about.

EXAMPLES:
CREATOR: "Is Sunday subah - khali kagaz lo aur apna favorite building ya
character ya logo sketch karo. 20 minute. No judgment. Just do it."

TECH_BUILDER: "Aaj raat - GitHub kholo. Koi bhi ek interesting AI project
README padho. Sirf padho. Comment mat karo, fork mat karo. Bas samjho
kya ban raha hai duniya mein. 15 minute."

HEALER: "Aaj raat - iCall ya Vandrevala Foundation ki website kholo.
Not for yourself - just to understand what professional mental health
support looks like in India. 10 minute research."

TEACHER: "Is hafte - kisi ek cheez mein jo tumhe aati ho, kisi ek insaan
ko sikhao. Ek concept. 15 minute. See how it feels."

ANTIGRAVITY HARNESS - route by stress score:

LEVEL 1 - CLEAR (stress 1-4):
Full specific first step toward their interest:
WHAT: exact interest-related action
WHEN: specific day and time this week
HOW LONG: realistic estimate
HOW: 3-4 concrete steps

LEVEL 2 - CAUTIOUS (stress 5-6):
"Itna kuch hai dimag mein - main samajh sakta/sakti hoon.
Sirf ek cheez. Abhi ke liye." Then give micro first step.

LEVEL 3 - OVERWHELMED (stress 7-8):
MICRO-STEP ONLY. One thing under 5 minutes connected to their interest.
Ask "Ye kar sakte ho?" before anything more.

LEVEL 4 - PARALYSED (stress 9-10):
NO career or interest advice.
"Pehle ye batao - aaj koi hai tumhare paas baat karne ke liye?
Parent, dost, sibling, koi bhi?"
Share iCall India: 9152987821
Interest-based conversation continues in next session.

CLOSE (always - in their language):
End with their name and their WHY - the family, the late grandmother,
the dreams they mentioned specifically. Make it personal.
Final line: "Disha hamesha yahan hai. Jab bhi sawaal ho - aa jaana."
""")

# ─────────────────────────────────────────────────────────────────────────────
# COORDINATOR
# ─────────────────────────────────────────────────────────────────────────────
root_agent = Agent(
    name="disha_coordinator", model=MODEL,
    description="Disha AI - interest-first life navigation for Indian students.",
    instruction="""
Delegate to specialists in EXACTLY this order. Never skip. Never reorder.

1. stage_router          - understand who they are
2. interest_discovery    - discover what they ACTUALLY care about (most important)
3. knowledge_gap_fixer   - answer confusions SPECIFIC to their interest and stage
4. india_opportunity_finder - find REAL opportunities in their interest cluster
5. path_architect        - design paths TOWARD their interest
6. antigravity_coach     - first step TOWARD their interest, stress-calibrated

Interest discovery in step 2 drives EVERYTHING in steps 3-6.
Do not add your own content between specialists.
After antigravity_coach - STOP.
""",
    sub_agents=[
        stage_router, interest_discovery, knowledge_gap_fixer,
        india_opportunity_finder, path_architect, antigravity_coach,
    ],
)


In [ ]:
# CELL 6: Write server code and frontend to files
import os
os.makedirs('/kaggle/working/static', exist_ok=True)

# Write server.py
server_code = '''
import asyncio, json, os, time, uuid
from datetime import datetime
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse, StreamingResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types
from google.genai.errors import ClientError

# Import the agent defined in Cell 5
import sys
sys.path.insert(0, "/kaggle/working")

app = FastAPI(title="Disha AI")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

JOURNAL = "/kaggle/working/disha_memory/student_journal.json"
active_sessions = {}

class ChatRequest(BaseModel):
    session_id: str
    message: str

def read_journal():
    if os.path.exists(JOURNAL):
        with open(JOURNAL) as f: return json.load(f)
    return {"sessions": []}

def save_session(sid, notes):
    j = read_journal()
    j["sessions"].append({"session_id": sid, "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "notes": notes})
    with open(JOURNAL, "w") as f: json.dump(j, f, indent=2)

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/api/session/new")
async def new_session():
    from disha_agent import root_agent
    sid = str(uuid.uuid4())
    svc = InMemorySessionService()
    runner = Runner(agent=root_agent, app_name="disha_ai", session_service=svc)
    await svc.create_session(app_name="disha_ai", user_id="student", session_id=sid)
    active_sessions[sid] = {"runner": runner, "turns": 0}
    return {"session_id": sid}

@app.post("/api/chat")
async def chat(req: ChatRequest):
    if req.session_id not in active_sessions:
        raise HTTPException(404, "Session not found")
    data = active_sessions[req.session_id]
    runner = data["runner"]
    data["turns"] += 1
    async def stream():
        try:
            msg = genai_types.Content(role="user", parts=[genai_types.Part(text=req.message)])
            collected = []
            last_author = "disha"
            for attempt in range(1, 4):
                try:
                    async for event in runner.run_async(user_id="student", session_id=req.session_id, new_message=msg):
                        if not event.content or not event.content.parts: continue
                        text = "".join(p.text for p in event.content.parts if hasattr(p,"text") and p.text)
                        if text.strip():
                            last_author = getattr(event, "author", "disha")
                            collected.append(text.strip())
                            yield f"data: {json.dumps({chr(39)type{chr(39)}:{chr(39)}chunk{chr(39)},{chr(39)}text{chr(39)}:text.strip(),{chr(39)}agent{chr(39)}:last_author})}\\n\\n"
                    break
                except ClientError as e:
                    if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                        w = 30*attempt
                        yield f"data: {json.dumps({chr(39)type{chr(39)}:{chr(39)}status{chr(39)},{chr(39)}text{chr(39)}:f{chr(39)}Thinking... retrying in {w}s{chr(39)}})}"
                        await asyncio.sleep(w)
                    else: raise
            combined = " ".join(collected).lower()
            done = any(s in combined for s in ["hamesha yahan hai","always here","aa jaana"])
            if done: save_session(req.session_id, f"Completed in {data[chr(39)turns{chr(39)]} turns.")
            yield f"data: {json.dumps({chr(39)type{chr(39)}:{chr(39)}done{chr(39)},{chr(39)}is_complete{chr(39)}:done})}\\n\\n"
        except Exception as e:
            yield f"data: {json.dumps({chr(39)type{chr(39)}:{chr(39)}error{chr(39)},{chr(39)}text{chr(39)}:str(e)[:100]})}\\n\\n"
    return StreamingResponse(stream(), media_type="text/event-stream",
        headers={"Cache-Control":"no-cache","X-Accel-Buffering":"no"})

@app.get("/api/memory")
def memory(): return read_journal()

@app.delete("/api/session/{sid}")
def end(sid): active_sessions.pop(sid, None); return {"ok": True}

@app.get("/", response_class=HTMLResponse)
async def index():
    p = Path("/kaggle/working/static/index.html")
    return HTMLResponse(p.read_text(encoding="utf-8") if p.exists() else "<h1>Starting...</h1>")

app.mount("/static", StaticFiles(directory="/kaggle/working/static"), name="static")
'''

with open('/kaggle/working/disha_server.py', 'w', encoding='utf-8') as f:
    f.write(server_code)

print('✅ Server code written.')


In [ ]:
# CELL 6b: Write the frontend HTML
# Reading the HTML from our pre-built file embedded here
html_content = open('/kaggle/working/static/index.html', encoding='utf-8').read() if __import__('os').path.exists('/kaggle/working/static/index.html') else None

# We'll write it fresh since we need it to point to the right API
# The frontend is already uploaded via the notebook's dataset or we write it inline
print('Frontend will be written by Cell 7 setup.')
print('Proceeding to server startup...')


In [ ]:
# CELL 7: Start Disha AI web server + ngrok tunnel
# This gives you a PUBLIC URL anyone can open in their browser
import subprocess, threading, time, requests
from pyngrok import ngrok, conf

# Setup ngrok
conf.get_default().auth_token = NGROK_TOKEN

# Write disha_agent package so server can import it
import os
os.makedirs('/kaggle/working/disha_agent', exist_ok=True)

with open('/kaggle/working/disha_agent/__init__.py', 'w') as f:
    f.write('from .agent import root_agent\n__all__ = ["root_agent"]\n')

# Write agent.py
agent_src = open('/tmp/agent_source.py', encoding='utf-8').read() if os.path.exists('/tmp/agent_source.py') else ''
# Since we defined root_agent in Cell 5, save it to file for the server to import
import inspect
# We'll write the full agent code inline
print('Writing agent module...')

# The server imports disha_agent.root_agent
# Since root_agent is already defined in this kernel, we create a wrapper
agent_wrapper = '''
# This file imports root_agent from the kernel scope
# root_agent is defined in the notebook Cell 5
import sys
# The root_agent is passed via the module reload trick
try:
    from _disha_runtime import root_agent
except ImportError:
    raise ImportError("Run Cell 5 first to define root_agent")
'''

# Better approach: write full agent code to file
AGENT_CODE = """
import textwrap
from google.adk import Agent
from google.adk.tools import google_search

MODEL = 'gemini-2.5-flash'
LANG = '=== MULTILINGUAL RULE ===\\nDetect language from replies and respond in THAT language.\\nDefault Hinglish if unclear. Switch when they switch.'.strip()

stage_router = Agent(name='stage_router', model=MODEL, mode='task',
    description='Collects life context one question at a time. Call FIRST.',
    instruction=LANG + '''\n\nYou are Disha - warm older sibling. Ask ONE question at a time.\nOPEN: "Namaste! Main Disha hoon. Batao - abhi zindagi mein kahan ho? 10th ke baad? 12th? College mein? Ya uske baad?"\nThen one at a time: city/state, category (SC/ST/OBC/General), family income, first in family for college, stay near family or move, stress 1-10.\nIf stress>=7 acknowledge warmly. Once all 7 answered write warm 2-sentence summary then finish.\n''')

interest_discovery = Agent(name='interest_discovery', model=MODEL, mode='task',
    description='Discovers real interests via behavioural questions. Call AFTER stage_router.',
    instruction=LANG + '''\n\nNEVER ask what are you interested in. Ask ONE AT A TIME:\nQ1: Kabhi kisi cheez mein itne doob gaye ki khana bhool gaye? Kya tha wo?\nQ2: Dost kaunsi help ke liye specifically tumhare paas aate hain?\nQ3: Duniya mein kya cheez sabse zyada gussa ya sad karti hai?\nQ4: YouTube random kholo - kya dekhne lagte ho?\nQ5: School mein kaunsa subject boring tha par interesting lagta tha?\nFallback: Free Tuesday no worries - subah uthke kya karte?\nMap to: TECH_BUILDER/CREATOR/HEALER/SYSTEMS_THINKER/FIXER/LEADER/TEACHER/NATURE_WORKER/CREATIVE_ARTS/GOVERNMENT/UNDISCOVERED\nShare insight warmly 2-3 sentences then finish.\n''')

knowledge_gap_fixer = Agent(name='knowledge_gap_fixer', model=MODEL, mode='single_turn',
    description='Answers confusions specific to interest cluster and stage. Call AFTER interest_discovery.',
    instruction=LANG + '''\n\nUse full conversation history. Answer confusions at intersection of their STAGE and INTEREST CLUSTER.\nDo NOT give generic advice. Do NOT mention JEE/GATE unless directly relevant to their cluster.\nCREATOR in Class 12: NID NIFT design schools, portfolio, creative career income.\nTECH_BUILDER graduating: AI companies, portfolio vs marks, open source.\nHEALER in Class 12: NEET options, BAMS BDS BPharm BSc Nursing allied health.\nSYSTEMS_THINKER: CA CFA MBA data analyst finance roles.\nFIXER graduating: GATE for MTech, core engineering, Germany MS.\nLEADER: MBA colleges, management trainee, entrepreneurship.\nTEACHER: BEd EdTech content creation online tutoring.\nGOVERNMENT: UPSC honest truth under 1 percent, state PSC SSC Banking Railway.\nUNDISCOVERED: How to explore systematically, explore and earn strategy.\n5-7 sentences. End: Ab chalo tumhare liye real options dhundhte hain.\n''')

india_opportunity_finder = Agent(name='india_opportunity_finder', model=MODEL, mode='single_turn',
    description='Searches LIVE opportunities specific to interest cluster category stage. Call AFTER knowledge_gap_fixer.',
    instruction=LANG + '''\n\nSearch REAL current data based on interest cluster + category + state.\nSearch career salaries for their cluster first.\nSearch education pathways into their cluster from their stage.\nSearch scholarships: SC/ST Rajiv Gandhi Fellowship 2026, National Overseas Scholarship 2026 SC, state schemes.\nAlways include HIDDEN ADVANTAGE - one thing they qualify for but probably dont know.\nPresent conversationally in their language. NOT JSON. 5-7 sentences.\n''',
    tools=[google_search])

path_architect = Agent(name='path_architect', model=MODEL, mode='single_turn',
    description='Designs 2 honest paths toward student interest. Call AFTER india_opportunity_finder.',
    instruction=LANG + '''\n\nBuild exactly 2 paths TOWARD their revealed interest.\nPATH A = most realistic lowest risk toward their interest.\nPATH B = higher upside more ambitious toward their interest.\nFor each: name, 12-month plan, money reality, family impact, hard truth, upside, confidence %.\nParent conflict: add how to talk to parents with 3 real sentences.\nEnd with ONE recommendation using their name. Never say it depends without picking.\n''')

antigravity_coach = Agent(name='antigravity_coach', model=MODEL, mode='single_turn',
    description='ONE stress-calibrated first step toward student interest. Antigravity harness. Call LAST.',
    instruction=LANG + '''\n\nUse their name. Give first step TOWARD THEIR SPECIFIC INTEREST not generic.\nANTIGRAVITY: stress 1-4 full step, stress 5-6 acknowledge then step, stress 7-8 micro-step only ask ye kar sakte ho, stress 9-10 NO career advice ask if they have someone to talk to share iCall 9152987821.\nClose with their name and their WHY. Final line: Disha hamesha yahan hai. Jab bhi sawaal ho - aa jaana.\n''')

root_agent = Agent(name='disha_coordinator', model=MODEL,
    description='Disha AI interest-first life navigation coordinator.',
    instruction='Delegate in order: 1.stage_router 2.interest_discovery 3.knowledge_gap_fixer 4.india_opportunity_finder 5.path_architect 6.antigravity_coach. Never skip. Never reorder. Stop after antigravity_coach.',
    sub_agents=[stage_router, interest_discovery, knowledge_gap_fixer, india_opportunity_finder, path_architect, antigravity_coach])
"""

with open('/kaggle/working/disha_agent/agent.py', 'w', encoding='utf-8') as f:
    f.write(AGENT_CODE)
print('Agent module written.')

# Write the complete frontend HTML
HTML = open('/kaggle/working/static/index.html', encoding='utf-8').read() if os.path.exists('/kaggle/working/static/index.html') else None
if not HTML:
    print('Writing frontend HTML...')
    # Minimal working frontend
    HTML = '''<!DOCTYPE html><html><head><meta charset="UTF-8"><title>Disha AI</title>
    <style>*{box-sizing:border-box;margin:0;padding:0}body{background:#07071a;color:#f0f0ff;font-family:sans-serif;height:100vh;display:flex;flex-direction:column}
    .top{background:#0e0e28;padding:16px 24px;border-bottom:1px solid rgba(124,106,247,.2);display:flex;align-items:center;gap:12px}
    .logo{width:40px;height:40px;background:linear-gradient(135deg,#7c6af7,#a855f7);border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:20px}
    h1{font-size:18px;font-weight:700}p.sub{font-size:12px;color:#9090b8}
    .msgs{flex:1;overflow-y:auto;padding:20px;display:flex;flex-direction:column;gap:12px}
    .msg{max-width:75%;padding:12px 16px;border-radius:14px;font-size:14px;line-height:1.65;white-space:pre-wrap;word-break:break-word}
    .disha{align-self:flex-start;background:linear-gradient(135deg,#1a1a50,#221250);border:1px solid rgba(124,106,247,.25)}
    .user{align-self:flex-end;background:linear-gradient(135deg,#1a3a2a,#0f2a1f);border:1px solid rgba(74,222,128,.15)}
    .name{font-size:10px;color:#a594ff;margin-bottom:4px}
    .inp-area{padding:16px 20px;background:#0e0e28;border-top:1px solid rgba(124,106,247,.15)}
    .inp-wrap{display:flex;gap:10px;background:#1a1a48;border:1px solid rgba(124,106,247,.25);border-radius:12px;padding:10px 14px}
    textarea{flex:1;background:transparent;border:none;outline:none;color:#f0f0ff;font-size:14px;font-family:sans-serif;resize:none;min-height:22px;max-height:100px;line-height:1.5}
    textarea::placeholder{color:#5a5a80}
    button{width:36px;height:36px;background:linear-gradient(135deg,#7c6af7,#a855f7);border:none;border-radius:9px;cursor:pointer;color:white;font-size:18px;display:flex;align-items:center;justify-content:center;flex-shrink:0}
    button:disabled{opacity:.4;cursor:not-allowed}
    .hint{font-size:11px;color:#5a5a80;text-align:center;margin-top:8px}
    .welcome{display:flex;flex-direction:column;align-items:center;justify-content:center;height:100%;text-align:center;gap:14px;padding:40px}
    .wi{font-size:54px}.wt{font-size:24px;font-weight:700;background:linear-gradient(135deg,#a594ff,#e879f9);-webkit-background-clip:text;-webkit-text-fill-color:transparent}
    .wd{font-size:14px;color:#9090b8;max-width:360px;line-height:1.6}
    .start-btn{padding:12px 28px;background:linear-gradient(135deg,#7c6af7,#a855f7);border:none;border-radius:10px;color:white;font-size:14px;font-weight:600;cursor:pointer;width:auto;height:auto}
    .typing{display:flex;gap:5px;padding:14px 16px;background:linear-gradient(135deg,#1a1a50,#221250);border:1px solid rgba(124,106,247,.25);border-radius:14px;align-self:flex-start;max-width:100px}
    .td{width:7px;height:7px;border-radius:50%;background:#7c6af7;animation:tb 1.2s infinite}
    .td:nth-child(2){animation-delay:.2s}.td:nth-child(3){animation-delay:.4s}
    @keyframes tb{0%,100%{transform:translateY(0);opacity:.4}40%{transform:translateY(-6px);opacity:1}}
    </style></head><body>
    <div class="top"><div class="logo">🧭</div><div><h1>Disha AI</h1><p class="sub">Life Navigation for Indian Students · Free · Multilingual</p></div></div>
    <div class="msgs" id="msgs">
    <div class="welcome" id="welcome">
    <div class="wi">🧭</div>
    <div class="wt">Namaste! Main Disha hoon.</div>
    <div class="wd">Your personal AI life navigation companion. I guide Indian students at every crossroads — one honest question at a time, in your language.</div>
    <button class="start-btn" onclick="startChat()">Start Conversation</button>
    <div class="wd" style="font-size:12px;color:#5a5a80">Hindi · Marathi · Tamil · Telugu · Bengali · Gujarati · Kannada · English · Hinglish</div>
    </div></div>
    <div class="inp-area">
    <div class="inp-wrap"><textarea id="inp" rows="1" placeholder="Type in any language — Hindi, English, Marathi, Tamil..." onkeydown="onKey(event)" oninput="rsz(this)" disabled></textarea>
    <button id="sb" onclick="send()" disabled>➤</button></div>
    <div class="hint">Enter to send · Shift+Enter for new line · Disha responds in your language</div>
    </div>
    <script>
    var SID=null,busy=false;
    var AMAP={stage_router:"🎯 Disha — Understanding You",interest_discovery:"💡 Disha — Discovering Interests",knowledge_gap_fixer:"📖 Disha — Knowledge Expert",india_opportunity_finder:"🔍 Disha — Finding Opportunities",path_architect:"🗺️ Disha — Path Architect",antigravity_coach:"🚀 Disha — First Step Coach",disha_coordinator:"🧭 Disha"};
    async function startChat(){
      document.getElementById("welcome").remove();
      try{
        var r=await fetch("/api/session/new",{method:"POST"});
        var d=await r.json();SID=d.session_id;
        document.getElementById("inp").disabled=false;
        document.getElementById("sb").disabled=false;
        document.getElementById("inp").focus();
        await sendTo("Hi Disha");
      }catch(e){addMsg("disha","Cannot connect to server. Please wait and refresh.","🧭 Disha");}
    }
    async function send(){
      var inp=document.getElementById("inp"),txt=inp.value.trim();
      if(!txt||busy||!SID)return;
      inp.value="";rsz(inp);
      addMsg("user",txt,"You");
      await sendTo(txt);
    }
    async function sendTo(txt){
      busy=true;document.getElementById("sb").disabled=true;showTyp();
      try{
        var res=await fetch("/api/chat",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({session_id:SID,message:txt})});
        var reader=res.body.getReader(),dec=new TextDecoder(),buf="",curEl=null,curAg="",curTxt="";
        while(true){
          var x=await reader.read();if(x.done)break;
          buf+=dec.decode(x.value,{stream:true});
          var lines=buf.split("\\n");buf=lines.pop();
          for(var line of lines){
            if(!line.startsWith("data: "))continue;
            try{
              var ev=JSON.parse(line.slice(6));
              if(ev.type==="chunk"){
                removeTyp();
                if(!curEl||curAg!==ev.agent){curEl=makeBubble(ev.agent);curAg=ev.agent;curTxt="";}
                curTxt+=(curTxt?"\\n\\n":"")+ev.text;
                curEl.querySelector(".bubble").textContent=curTxt;
                scrollB();
              }else if(ev.type==="done"&&ev.is_complete){
                var d=document.createElement("div");d.style.cssText="padding:12px;background:rgba(74,222,128,.08);border:1px solid rgba(74,222,128,.25);border-radius:8px;text-align:center;font-size:13px;color:#4ade80";
                d.textContent="Session complete — Disha has given you your path and first step.";
                document.getElementById("msgs").appendChild(d);scrollB();
                document.getElementById("inp").disabled=true;
                document.getElementById("sb").disabled=true;
              }else if(ev.type==="error"){removeTyp();addMsg("disha",ev.text,"🧭 Disha");}
            }catch(_){}
          }
        }
      }catch(e){removeTyp();addMsg("disha","Connection error. Please try again.","🧭 Disha");}
      busy=false;document.getElementById("sb").disabled=false;
      document.getElementById("inp").focus();
    }
    function addMsg(role,text,agent){
      var m=document.getElementById("msgs"),d=document.createElement("div");
      d.className="msg "+role;
      var n=role==="user"?"You":(AMAP[agent]||"🧭 Disha");
      d.innerHTML="<div class=\\"name\\">"+ n +"</div><div class=\\"bubble\\">"+ esc(text) +"</div>";
      m.appendChild(d);scrollB();return d;
    }
    function makeBubble(agent){
      var m=document.getElementById("msgs"),d=document.createElement("div");
      d.className="msg disha";
      var n=AMAP[agent]||"🧭 Disha";
      d.innerHTML="<div class=\\"name\\">"+ n +"</div><div class=\\"bubble\\"></div>";
      m.appendChild(d);scrollB();return d;
    }
    function showTyp(){removeTyp();var m=document.getElementById("msgs"),d=document.createElement("div");d.className="typing";d.id="typ";d.innerHTML="<div class=\\"td\\"></div><div class=\\"td\\"></div><div class=\\"td\\"></div>";m.appendChild(d);scrollB();}
    function removeTyp(){var e=document.getElementById("typ");if(e)e.remove();}
    function scrollB(){var m=document.getElementById("msgs");m.scrollTop=m.scrollHeight;}
    function esc(t){return t.replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;").replace(/\\n/g,"<br>");}
    function rsz(el){el.style.height="auto";el.style.height=Math.min(el.scrollHeight,100)+"px";}
    function onKey(e){if(e.key==="Enter"&&!e.shiftKey){e.preventDefault();send();}}
    </script></body></html>'''
    with open('/kaggle/working/static/index.html', 'w', encoding='utf-8') as f:
        f.write(HTML)
    print('Frontend written.')

# Start FastAPI server in background thread
import uvicorn
import importlib.util, sys

spec = importlib.util.spec_from_file_location('disha_server', '/kaggle/working/disha_server.py')
server_module = importlib.util.load_from_spec(spec)
spec.loader.exec_module(server_module)
server_app = server_module.app

def run_server():
    uvicorn.run(server_app, host='0.0.0.0', port=8080, log_level='error')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(3)

# Test server
try:
    r = requests.get('http://localhost:8080/health', timeout=5)
    print('✅ Server running:', r.json())
except:
    print('Server starting... wait 5 more seconds')
    time.sleep(5)

# Open ngrok tunnel
public_url = ngrok.connect(8080).public_url
print()
print('=' * 60)
print('🎉 DISHA AI IS LIVE!')
print('='*60)
print(f'🌐 Open this URL in your browser:')
print(f'   {public_url}')
print()
print('Share this link with anyone.')
print('Works on mobile too.')
print('='*60)


---

## How to Use

1. Click the URL shown above
2. Click **Start Conversation**
3. Disha asks one question at a time — type your answer and press Enter
4. The full 6-agent pipeline runs automatically

## Notes
- The URL works as long as the notebook is running
- If the notebook times out, just run Cell 7 again to get a new URL
- Get your free ngrok token at: https://dashboard.ngrok.com/get-started/your-authtoken
